# 7. Delta Sharing — KPI financieros | Banking Intelligence Colombia

Adaptación del notebook de ejemplo `5. DeltaSharing.ipynb` al catálogo `catalog_au`. Ejecutar paso a paso en un notebook de Azure Databricks con Unity Catalog.

**Alcance:** activos, pasivos, patrimonio y crecimiento interanual por entidad y enero de 2024–2026. Incluye entidades agregadas del origen; **no** constituye todavía un universo depurado de bancos. Los valores y unidades de la SFC están pendientes de validación contable y de unidades.


## 1. Explorar los datos de origen


In [0]:
%sql
SELECT fecha_corte, COUNT(*) AS registros FROM catalog_au.golden.fact_indicadores_financieros GROUP BY fecha_corte ORDER BY fecha_corte;


## 2. Publicar una tabla Delta de KPI para compartir

Delta Sharing compartirá una **tabla Delta física**, no la vista de Power BI. La siguiente consulta utiliza los códigos de conceptos confirmados y conserva valores en unidades originales (sin conversión a millones). No sumar los conceptos hijos a estos totales.


In [0]:
%sql
CREATE OR REPLACE TABLE catalog_au.golden.kpis_bancarios_sharing
USING DELTA
AS
WITH base AS (
  SELECT e.codigo_entidad, e.nombre_entidad, e.tipo_entidad,
         f.fecha_corte,
         MAX(CASE WHEN c.codigo_concepto = 'SFC_FILA_0014' THEN f.valor_indicador END) AS activos,
         MAX(CASE WHEN c.codigo_concepto = 'SFC_FILA_0101' THEN f.valor_indicador END) AS pasivos,
         MAX(CASE WHEN c.codigo_concepto = 'SFC_FILA_0168' THEN f.valor_indicador END) AS patrimonio
  FROM catalog_au.golden.fact_indicadores_financieros f
  JOIN catalog_au.golden.dim_entidad e ON f.sk_entidad = e.sk_entidad
  JOIN catalog_au.golden.dim_concepto_financiero c ON f.sk_concepto = c.sk_concepto
  WHERE c.codigo_concepto IN ('SFC_FILA_0014','SFC_FILA_0101','SFC_FILA_0168')
  GROUP BY e.codigo_entidad, e.nombre_entidad, e.tipo_entidad, f.fecha_corte
),
serie AS (
 SELECT *, LAG(activos) OVER (PARTITION BY codigo_entidad ORDER BY fecha_corte) AS activos_anio_anterior
 FROM base
)
SELECT codigo_entidad, nombre_entidad, tipo_entidad, fecha_corte,
       activos, pasivos, patrimonio, activos_anio_anterior,
       CASE WHEN activos_anio_anterior IS NOT NULL AND activos_anio_anterior <> 0
            THEN CAST(((activos - activos_anio_anterior) / activos_anio_anterior) * 100 AS DECIMAL(20,6))
       END AS crecimiento_activos_interanual_pct,
       'UNIDAD_PENDIENTE_VALIDACION' AS estado_unidad
FROM serie;


## 3. Validar la tabla de KPI

Verifica cobertura, duplicados por entidad/fecha y la presencia de agregados sectoriales antes de habilitar el acceso externo.


In [0]:
%sql
SELECT COUNT(*) AS filas,
       COUNT(DISTINCT concat_ws('|',codigo_entidad,cast(fecha_corte AS STRING))) AS claves_entidad_fecha,
       COUNT(DISTINCT fecha_corte) AS periodos,
       SUM(CASE WHEN activos IS NULL OR pasivos IS NULL OR patrimonio IS NULL THEN 1 ELSE 0 END) AS balances_incompletos
FROM catalog_au.golden.kpis_bancarios_sharing;


In [0]:
%sql
SELECT * FROM catalog_au.golden.kpis_bancarios_sharing ORDER BY fecha_corte, activos DESC LIMIT 20;


## 4. Crear el share y añadir la tabla

Ejecutar solo cuando hayas revisado los KPI. Requiere permisos de Unity Catalog para crear y gestionar shares. `WITH HISTORY` habilita compartir historial cuando la configuración y permisos lo permiten.


In [0]:
%sql
CREATE SHARE IF NOT EXISTS BIC_KPIS_SHARE;


In [0]:
%sql
SHOW SHARES;


In [0]:
%sql
ALTER SHARE BIC_KPIS_SHARE ADD TABLE catalog_au.golden.kpis_bancarios_sharing WITH HISTORY;


## 5. Registrar destinatario externo (opcional)

**No ejecutes este paso hasta decidir quién recibirá los datos.** El ejemplo original usa `CREATE RECIPIENT` y `GRANT SELECT`. Para un destinatario externo con token, Databricks puede generar credenciales que deben distribuirse por un canal seguro. Reemplaza el nombre de ejemplo por uno aprobado. No pegues tokens ni enlaces de activación en chats o notebooks.


In [0]:
%sql
-- Ejecutar solo tras aprobar el destinatario y el método de autenticación:
-- CREATE RECIPIENT BIC_SOCIO_EXTERNO;
-- GRANT SELECT ON SHARE BIC_KPIS_SHARE TO RECIPIENT BIC_SOCIO_EXTERNO;
-- SHOW GRANTS ON SHARE BIC_KPIS_SHARE;


## 6. Notas de operación

- Si vuelves a ejecutar `ALTER SHARE ... ADD TABLE` sobre una tabla ya incluida, puede fallar por duplicidad: comprueba antes el contenido del share.
- `CREATE OR REPLACE TABLE` reconstruye los KPI y no debe ejecutarse sin revisar el impacto en lectores externos e historial. Para producción, usar cargas incrementales y gobierno de cambios.
- La participación de mercado **no** se publica todavía porque falta delimitar y validar el universo de bancos individuales; tampoco se publican razones financieras sin unidades verificadas.
- Evita compartir `fact_indicadores_financieros` y dimensiones completas si el destinatario solo necesita estos KPI.
